<a href="https://colab.research.google.com/github/Chenuka-Garusinghe/LLM-ICL-OOD-Honours/blob/main/sata-project/notebooks/01_tableshift_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: TableShift Setup

**Purpose**: Load TableShift, select the candidate datasets, preprocess, verify OOD splits exist and show meaningful shift gaps.

**Real arm. No training. This is the pipeline smoke test.**

## Why this notebook exists (thesis framing)

This notebook is the data foundation for **RQ1**: *how well do frozen LLMs perform OOD in-context generalisation on structured tabular decision tasks under controlled distribution shifts?* Everything downstream (Notebooks 02, 03, 07, 08's real-arm figures) reads from what this notebook produces.

The literature review (`Lit-review.pdf`, Section 2.1.3) motivates using **TableShift** (Gardner et al., NeurIPS 2023) specifically because it's one of the only tabular benchmarks that pairs each prediction task with a *naturally occurring* shift (geography, demographics, time) rather than a synthetic split, and because it ships a metric suite (OTDD for covariate shift, FDD for concept shift, base-rate L2 for label shift) that maps directly onto the classic dataset-shift taxonomy (Moreno-Torres et al. 2012; Storkey 2009) — covariate shift, prior-probability (label) shift, and concept shift. This project doesn't currently compute TableShift's OTDD/FDD metrics directly (that's a possible extension), but the ID-vs-OOD split structure it provides is exactly the shift structure RQ1 needs.

**Why real-world tabular data at all, and not just synthetic tasks?** Section 2.4.2 of the lit review argues tabular data is a uniquely *auditable* testbed for shortcut learning: features have semantic names and known causal roles (e.g. ZIP code as a proxy for race in the ACS benchmark), so a spurious correlation can be identified and named, unlike texture-bias in vision benchmarks. Real TableShift datasets ground RQ1's headline claim ("does the problem exist?") in a setting practitioners actually care about; the synthetic generator built in Notebook 04 is what supplies *ground truth* for causal reliance, which no real dataset can give us (see that notebook for why).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cwd /content/LLM-ICL-OOD-Honours/sata-project/notebooks

UsageError: Line magic function `%cwd` not found.


In [4]:
REPO_CLONE_DIR = '/content/LLM-ICL-OOD-Honours'
PROJECT_DIR = f'{REPO_CLONE_DIR}/sata-project'
DRIVE_BASE = '/content/drive/MyDrive/sata-project'

get_ipython().system(f"test -d {REPO_CLONE_DIR}/.git || git clone https://github.com/Chenuka-Garusinghe/LLM-ICL-OOD-Honours.git {REPO_CLONE_DIR}")
get_ipython().system(f"bash {PROJECT_DIR}/scripts/deploy_to_colab.sh")

import os

os.environ['HF_HOME'] = f'{DRIVE_BASE}/.cache/huggingface'

# Symlink TableShift cache dirs to Drive so extraction output lands directly on Drive
os.makedirs(f'{DRIVE_BASE}/tableshift_raw_cache', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/tableshift_cache', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.path.lexists(f'{PROJECT_DIR}/data/tableshift_raw_cache') or os.symlink(f'{DRIVE_BASE}/tableshift_raw_cache', f'{PROJECT_DIR}/data/tableshift_raw_cache')
os.path.lexists(f'{PROJECT_DIR}/data/tableshift_cache') or os.symlink(f'{DRIVE_BASE}/tableshift_cache', f'{PROJECT_DIR}/data/tableshift_cache')
os.chdir(f'{PROJECT_DIR}/notebooks')
print("cwd now:", os.getcwd())


Cloning into '/content/LLM-ICL-OOD-Honours'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 121 (delta 53), reused 104 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 650.19 KiB | 10.84 MiB/s, done.
Resolving deltas: 100% (53/53), done.
== Step 1/5: check Google Drive is mounted ==
Drive OK: /content/drive/MyDrive/sata-project
== Step 2/5: get the project onto the VM ==
Repo already present at /content/LLM-ICL-OOD-Honours -- pulling latest.
Already up to date.
== Step 3/5: install Python dependencies into the kernel env ==
HF_HOME set to /content/drive/MyDrive/sata-project/.cache/huggingface for this shell -- see note below about the kernel process.
== Step 4/5: check TableShift raw cache ==
MISSING datasets: acsincome acspubcov brfss_diabetes anes
Run scripts/extract_tableshift_cache.py from a SEPARATE, throwaway environment
(your Mac or Gadi -- NOT this Cola

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import load_config, set_seed, resolve_path

config = load_config()
set_seed(config.seed_accuracy[0])

## Step 1: Install / import TableShift

**`tableshift` is not a dependency of this project's environment, and never will be.** It hard-pins `numpy==1.23.5` / `ray==2.2`, and its `xport` dependency breaks outright on `pandas>=3` — all incompatible with this project's modern stack (torch 2.x, vllm, current pandas/numpy/sklearn). Installing it into the same environment as everything else means re-fighting that version conflict indefinitely.

Instead: clone `github.com/mlfoundations/tableshift` and `pip install -e . --no-deps` (plus its runtime deps) into a **separate, throwaway environment** — conda is the natural choice, but any isolated environment works. From that environment, run `python scripts/extract_tableshift_cache.py` **once** — it downloads/loads each candidate dataset and dumps `train`/`test_id`/`test_ood` as plain parquet files under `data/tableshift_raw_cache/{dataset_name}/`. See that script's docstring for the full instructions, including why `anes` needs a manual download (Step 2 below).

After that one-time extraction, **this project's own environment never imports `tableshift`** — `src/data/tableshift_loader.py::load_tableshift_splits` only ever reads the cached parquet files. The throwaway environment can be deleted once extraction succeeds.

In [6]:
from src.data.tableshift_loader import CANDIDATE_DATASETS

CANDIDATE_DATASETS

['acsincome', 'acspubcov', 'brfss_diabetes', 'anes']

## Step 2: Select datasets

Candidates ranked by published shift gap and public accessibility:
- ACS Income (geographic shift)
- ACS Public Coverage (demographic shift)
- BRFSS Diabetes (temporal/geographic shift)
- ANES Voting (temporal shift)

Selection criteria: public access, binary classification, <=15 usable features after reduction, nontrivial published shift gap.

**Decision**: all 4 candidates verified to load end-to-end via `load_tableshift_splits` (from the cached parquet files — see Step 1). `anes` is TableShift's one `OfflineDataSource` — it can't auto-download; from the isolated extraction environment (Step 1), it needs the Time Series Cumulative Data File manually downloaded from electionstudies.org and placed as `anes_timeseries_cdf_csv_20220916.csv` under `data/tableshift_cache/` (the *download* cache `scripts/extract_tableshift_cache.py` passes to `tableshift.get_dataset(cache_dir=...)` — not the `data/tableshift_raw_cache/` parquet output this project's own environment reads from). tableshift hardcodes this exact filename/date regardless of which release you actually download — a newer release works as long as the `VCF*` columns `tableshift.datasets.anes.ANES_FEATURES` expects are present, which we verified.

In [7]:
from src.data.tableshift_loader import SELECTED_DATASETS

# brfss_diabetes, acsincome, acspubcov, anes — all 4 verified end-to-end.
# Note: the spec's Notebook 01 originally called for 3 datasets; we're
# keeping all 4 available since anes turned out to be usable too. Trim
# this list back to 3 here if you'd rather match the spec's original scope.
SELECTED_DATASETS

['brfss_diabetes', 'acsincome', 'acspubcov', 'anes']

## Step 3: Preprocessing per dataset

- Load train / ID-test / OOD-test splits via TableShift API.
- Feature reduction: top 10-15 features by mutual information with the label.
- Missing values: mode imputation (categorical), median (continuous).
- Demo pool: 256 rows from training split, stratified by label, fixed across all conditions/seeds.
- Test sets: 500 ID-test + 500 OOD-test rows.
- Save as parquet.

**Why a fixed 256-row demo pool, sampled once per dataset?** This pool is the *support set* every demo-selection condition (Notebook 02) draws from — it has to be identical across conditions and seeds so that a comparison between, say, random-k and counter-spurious diversity isn't confounded by drawing from different candidate rows. This mirrors how the Bayesian-inference account of ICL (Xie et al. 2022, Lit-review §2.3.1) frames the demonstration set as *evidence the model conditions its prediction on* — if the evidence pool itself changes between conditions, differences in downstream accuracy could just be measuring "who got luckier rows" rather than "which selection strategy is better."

**Why top-10 features by mutual information, not all of them?** Two reasons. First, the spec's selection criteria cap usable features at ≤15 so a serialised row stays a reasonable prompt length. Second — and this only matters once you get to Notebook 06 — SATA (Notebook 05) is meta-trained exclusively on synthetic tasks with a fixed `n_features=10`, so real datasets are reduced to the *same* dimensionality if SATA is ever asked to score real-arm demonstrations (a stretch goal noted in the spec). `config.generator.n_features` is reused here deliberately, not coincidentally.

In [21]:
!git clone --quiet https://github.com/mlfoundations/tableshift.git /content/tableshift-src 2>/dev/null; echo "tableshift-src ready: $(test -d /content/tableshift-src && echo yes || echo no)"

tableshift-src ready: yes


In [27]:
!python3 --version
!pip install -q uv
!uv venv --python 3.10 /content/tableshift-venv
!uv pip install --python /content/tableshift-venv/bin/python --no-deps -e /content/tableshift-src
!uv pip install --python /content/tableshift-venv/bin/python numpy==1.23.5 ray==2.2.0 xport pandas scikit-learn setuptools==65.5.0

Python 3.13.15
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: /content/tableshift-venv
? A virtual environment already exists at `/content/tableshift-venv`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt^C
Using Python 3.10.12 environment at: /content/tableshift-venv
Resolved 1 package in 638ms
Prepared 1 package in 268ms
Uninstalled 1 package in 0.48ms
Installed 1 package in 1ms
 ~ tableshift==0.1 (from file:///content/tableshift-src)
Using Python 3.10.12 environment at: /content/tableshift-venv
Resolved 36 packages in 119ms
Prepared 1 package in 186ms
Uninstalled 1 package in 6ms
Installed 1 package in 6ms
 - setuptools==84.0.0
 + setuptools==65.5.0


In [28]:
!source /content/tableshift-venv/bin/activate

In [29]:
!/content/tableshift-venv/bin/python /content/LLM-ICL-OOD-Honours/sata-project/scripts/extract_tableshift_cache.py

Traceback (most recent call last):
  File "/content/LLM-ICL-OOD-Honours/sata-project/scripts/extract_tableshift_cache.py", line 85, in <module>
    extract_dataset(name)
  File "/content/LLM-ICL-OOD-Honours/sata-project/scripts/extract_tableshift_cache.py", line 66, in extract_dataset
    _patch_xport_underscore_fields()
  File "/content/LLM-ICL-OOD-Honours/sata-project/scripts/extract_tableshift_cache.py", line 52, in _patch_xport_underscore_fields
    if getattr(_xport_v56.namedtuple, "_extract_script_patch", False):
AttributeError: module 'xport.v56' has no attribute 'namedtuple'


In [8]:
from tqdm import tqdm

from src.data.tableshift_loader import (
    load_tableshift_splits,
    select_top_features,
    impute_missing,
    build_demo_pool,
    save_dataset_artifacts,
)

dataset_artifacts = {}

for dataset_name in tqdm(SELECTED_DATASETS, desc="Datasets"):
    splits = load_tableshift_splits(dataset_name)
    feature_cols = select_top_features(splits['train'], n_features=config.generator.n_features)

    train_imputed = impute_missing(splits['train'], feature_cols)
    test_id_imputed = impute_missing(splits['test_id'], feature_cols)
    test_ood_imputed = impute_missing(splits['test_ood'], feature_cols)

    train_pool = build_demo_pool(train_imputed, config.pool_size, seed=config.seed_accuracy[0])

    test_id = test_id_imputed.sample(
        n=min(config.test_rows_id, len(test_id_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)
    test_ood = test_ood_imputed.sample(
        n=min(config.test_rows_ood, len(test_ood_imputed)), random_state=config.seed_accuracy[0]
    ).reset_index(drop=True)

    # TableShift labels are already binary 0/1; stringify for use as LLM label tokens.
    label_tokens = [str(v) for v in sorted(splits['train']['label'].unique())]

    save_dataset_artifacts(
        dataset_name=dataset_name,
        train_pool=train_pool[feature_cols + ['label']],
        test_id=test_id[feature_cols + ['label']],
        test_ood=test_ood[feature_cols + ['label']],
        feature_list=feature_cols,
        label_tokens=label_tokens,
        out_root=resolve_path(config.paths.data_real),
    )

    dataset_artifacts[dataset_name] = {
        'feature_cols': feature_cols,
        'label_tokens': label_tokens,
        'pool_size': len(train_pool),
        'test_id_size': len(test_id),
        'test_ood_size': len(test_ood),
    }
    tqdm.write(f"{dataset_name}: pool={len(train_pool)} id_test={len(test_id)} ood_test={len(test_ood)} "
               f"labels={label_tokens} features={feature_cols}")

dataset_artifacts

Datasets:   0%|          | 0/4 [00:00<?, ?it/s]


FileNotFoundError: /content/LLM-ICL-OOD-Honours/sata-project/data/tableshift_raw_cache/brfss_diabetes/train.parquet not found. Run `python scripts/extract_tableshift_cache.py brfss_diabetes` from a separate, isolated environment with `tableshift` installed first — this project's own environment never installs tableshift. See that script's docstring.

## Step 4: Serialisation template

See `src/data/serialisation.py::serialise_row`. Feature order is fixed alphabetically per dataset and recorded in `feature_list.json` — never randomise it.

### Why serialise to text at all? (SATA vs. TabPFN — resolving the lit-review question)

The lit review (§2.4.1) draws a hard line between two ways of doing tabular in-context learning, and it matters for understanding what this whole project *is*:

- **Purpose-trained tabular ICL (TabPFN, Hollmann et al.)**: a transformer pretrained *exclusively* on synthetic tabular tasks. It takes the support set as raw numeric input and **is itself the classifier** — one forward pass produces the prediction. It has no language-modelling capability and never sees text.
- **General-purpose LLM ICL via prompt serialisation** — what this project does: a tabular row is converted into a natural-language string (`"Age: 45; Income: 80000 -> Approved"`), placed in the prompt as a demonstration, and a frozen, general-purpose LLM (Llama-3.1-8B / Qwen2.5-7B) predicts the query's label the same way it would answer any other few-shot prompt.

**This is why serialisation exists as a step at all** — it's the mechanism by which a tabular row becomes something a language model's pretrained ICL circuitry can act on. It also means this project's failure modes are different from TabPFN's: TabPFN fails when a task falls outside its synthetic-prior's coverage; a general-purpose LLM fails according to whatever spurious correlations and biases its *pretraining corpus* happened to encode. That's precisely the failure mode the shortcut-learning literature (Geirhos et al. 2020, §2.2) describes, and it's why demonstration design — not model retraining — is the lever this project pulls.

**Where does SATA (Notebook 05) fit?** SATA is neither of the above. It doesn't classify anything. It's a small transformer that sits *before* this serialisation step in the pipeline and decides *which* demo rows get serialised into the prompt in the first place — the frozen LLM still does 100% of the actual prediction via the ICL mechanism this notebook is setting up. See Notebook 05's intro for the full architecture rationale.

In [ ]:
import json

import pandas as pd

from src.data.serialisation import serialise_row, ordered_feature_names

sample_dataset = SELECTED_DATASETS[0]
sample_dir = resolve_path(config.paths.data_real) / sample_dataset
sample_pool = pd.read_parquet(sample_dir / 'train_pool.parquet')
feature_list = json.load(open(sample_dir / 'feature_list.json'))
label_tokens = json.load(open(sample_dir / 'label_tokens.json'))

row = sample_pool.iloc[0]
ordered_feats = ordered_feature_names({f: row[f] for f in feature_list})
demo_text = serialise_row({f: row[f] for f in ordered_feats}, label=str(row['label']))
query_text = serialise_row({f: row[f] for f in ordered_feats})

print(demo_text)
print(query_text)

BMI5: -0.5544626934239348; BMI5CAT_20: 1.0; BMI5CAT_40: 0.0; HIGH_BLOOD_PRESS_10: 0.0; HIGH_BLOOD_PRESS_20: 1.0; MICHD_10: 0.0; PHYSHLTH: -0.4823046145944116; SMOKE100_20: 1.0; TOLDHI_10: 0.0; VEG_ONCE_PER_DAY_20: 0.0 -> 1.0
BMI5: -0.5544626934239348; BMI5CAT_20: 1.0; BMI5CAT_40: 0.0; HIGH_BLOOD_PRESS_10: 0.0; HIGH_BLOOD_PRESS_20: 1.0; MICHD_10: 0.0; PHYSHLTH: -0.4823046145944116; SMOKE100_20: 1.0; TOLDHI_10: 0.0; VEG_ONCE_PER_DAY_20: 0.0 ->


## Step 5: Pilot run

Zero-shot and random-8 on **one** dataset with **one** model. Verify the pipeline end-to-end: serialisation -> prompt -> vLLM -> prediction -> logprobs -> accuracy. This is a smoke test, not a result.

**What "Gate 1" actually checks.** Per the spec's week-by-week plan, Gate 1 asks: *does vanilla ICL degrade OOD on real data at all?* This matters because RQ1's own success criterion (Lit-review §3, Task 2) is explicit: RQ1 succeeds only if vanilla ICL with random demonstrations shows *measurable* OOD degradation (via R-AUC and shift gap) on a majority of benchmark settings. If frozen LLMs already generalised fine OOD with no demonstration design at all, there would be no shortcut-learning problem for the rest of the project (demonstration diversity protocols, SATA) to solve — the whole downstream research programme is conditional on this gate passing on real data, not just in the synthetic generator (which is calibrated by construction in Notebook 04 to exhibit shortcut learning).

In [ ]:
from tqdm import tqdm

from src.inference.llm_runner import VLLMRunner, get_confidence
from src.inference.prompts import build_classification_prompt
from src.selection.random_select import select as random_select

try:
    import vllm  # noqa: F401
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False
    print("vLLM not installed in this environment — skipping the pilot run. "
          "This cell needs a GPU box with vllm + the model weights available; "
          "run it there before Gate 1.")

if VLLM_AVAILABLE:
    PILOT_N_QUERIES = 5
    pilot_dataset = SELECTED_DATASETS[0]
    pilot_dir = resolve_path(config.paths.data_real) / pilot_dataset
    pilot_pool = pd.read_parquet(pilot_dir / 'train_pool.parquet')
    pilot_test = pd.read_parquet(pilot_dir / 'test_id.parquet').head(PILOT_N_QUERIES)
    pilot_features = json.load(open(pilot_dir / 'feature_list.json'))
    pilot_label_tokens = tuple(json.load(open(pilot_dir / 'label_tokens.json')))
    task_description = f"the '{pilot_dataset}' outcome"

    runner = VLLMRunner(config.base_llms[0].path, **vars(config.vllm))

    pilot_rows = []
    for query_id, (_, query) in tqdm(list(enumerate(pilot_test.iterrows())), desc="Pilot queries"):
        ordered_feats = ordered_feature_names({f: query[f] for f in pilot_features})
        query_line = serialise_row({f: query[f] for f in ordered_feats})

        for method, k in [('zero_shot', 0), ('random', config.k_primary)]:
            if k == 0:
                demo_ids, demo_lines = [], []
            else:
                demo_ids = random_select(pilot_pool, query, k=k, seed=config.seed_accuracy[0])
                demo_lines = [
                    serialise_row(
                        {f: pilot_pool.loc[i, f] for f in ordered_feature_names({f: pilot_pool.loc[i, f] for f in pilot_features})},
                        label=str(pilot_pool.loc[i, 'label']),
                    )
                    for i in demo_ids
                ]
            prompt = build_classification_prompt(task_description, pilot_label_tokens, demo_lines, query_line)
            pred = runner.batch_predict([prompt], pilot_label_tokens)[0]
            pilot_rows.append({
                'method': method, 'query_id': query_id, 'k': k,
                'prediction': pred.prediction, 'label': str(query['label']),
                'confidence': pred.confidence, 'logprob_0': pred.logprob_0, 'logprob_1': pred.logprob_1,
            })

    pilot_df = pd.DataFrame(pilot_rows)
    pilot_df

/Users/chenuka/Documents/USYD/LLM-ICL-OOD-Honours/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


INFO 08-25 12:34:50 [__init__.py:216] Automatically detected platform cpu.


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

## Output

- `data/real/{dataset_name}/train_pool.parquet` (256 rows)
- `data/real/{dataset_name}/test_id.parquet` (500 rows)
- `data/real/{dataset_name}/test_ood.parquet` (500 rows)
- `data/real/{dataset_name}/feature_list.json`
- `data/real/{dataset_name}/label_tokens.json`